In [4]:
import csv
import os

### Pasul 1 - Incarcare date

In [5]:
crtDir = os.getcwd()
fileName = os.path.join(crtDir, "data", "reviews_mixed.csv")

data = []
with open(fileName, encoding="latin-1") as csv_file:
    csv_reader = csv.reader(csv_file, delimiter=',')
    line_count = 0
    for row in csv_reader:
        if line_count == 0:
            dataNames = row
        else:
            data.append(row)
        line_count += 1

# Luam doar mesajele
inputs = [data[i][0] for i in range(len(data))][:100]

# Luam doar etichetele
outputs = [data[i][1] for i in range(len(data))][:100]

labelNames = list(set(outputs))

print(inputs[:2])
print(outputs[:2])

['The rooms are extremely small, practically only a bed.', 'Room safe did not work.']
['negative', 'negative']


### Pasul 2 - Impartirea datelor in set de antrenare si set de testare

In [6]:
import numpy as np
np.random.seed(5)
noSamples = len(inputs)
indexes = [i for i in range(noSamples)]
trainSample = np.random.choice(indexes, int(0.8 * noSamples), replace=False)
testSample = [i for i in indexes if i not in trainSample]

trainInputs = [inputs[i] for i in trainSample]
trainOutputs = [outputs[i] for i in trainSample]
testInputs = [inputs[i] for i in testSample]
testOutputs = [outputs[i] for i in testSample]

### Pasul 3 - extragem caracteristici

In [7]:
# Bag Of Words - reprezinta fiecare mesaj ca un vector de frecventa a cuvintelor
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
trainFeatures = vectorizer.fit_transform(trainInputs) # fit_transform - invata vocabularul din datele de antrenare si numara de cate ori apare fiecare cuvant in fiecare mesaj din setul de antrenare, rezultand o matrice de dimensiune (numar_mesaje, numar_cuvinte_unice)
testFeatures = vectorizer.transform(testInputs) # transform - numara de cate ori apare fiecare cuvant in fiecare mesaj din setul de testare, folosind vocabularul invatat din datele de antrenare

# vocabulary size
print("Vocabulary size:", len(vectorizer.vocabulary_), " unique words")

# numarul de mesaje din setul de antrenare
print("Number of training samples:", len(trainInputs))

# shape-ul matricei de caracteristici pentru setul de antrenare
print("Shape of training features:", trainFeatures.shape)

print("Some words in the vocabulary:", vectorizer.get_feature_names_out()[-20:])

print("Feature example:", trainFeatures.toarray()[3])

Vocabulary size: 341  unique words
Number of training samples: 80
Shape of training features: (80, 341)
Some words in the vocabulary: ['was' 'wasn' 'water' 'we' 'wear' 'well' 'were' 'wet' 'which' 'whole'
 'window' 'windows' 'winter' 'with' 'work' 'working' 'workout' 'would'
 'you' 'your']
Feature example: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 2 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0]


### Pasul 4 - Antrenarea unui model de retea neuronala artificiala (ANN) pentru clasificare

In [8]:
import numpy as np
class ANN:
    def __init__(self, features, labels, hidden):
        self.features = features # numarul de caracteristici de intrare
        self.labels = labels # numarul de clase
        self.hidden = hidden # numarul de neuroni din stratul ascuns

        self.w1_matrix = 0.01 * np.random.rand(self.features, self.hidden) # matricea de greutati pentru stratul de intrare
        self.b1_matrix = np.zeros((1, self.hidden)) # matricea de bias pentru stratul de intrare
        self.w2_matrix = 0.01 * np.random.rand(self.hidden, self.labels) # matricea de greutati pentru stratul de iesire
        self.b2_matrix = np.zeros((1, self.labels)) # matricea de

    def forward_propagation(self, features):
        z1 = np.dot(features, self.w1_matrix) + self.b1_matrix # calculam activarea pentru stratul ascuns
        z1 = np.maximum(0, z1) # aplicam functia de activare ReLU
        z2 = np.dot(z1, self.w2_matrix) + self.b2_matrix # calculam activarea pentru stratul de iesire
        exp_scores = np.exp(z2 - np.max(z2, axis=1, keepdims=True)) # aplicam functia softmax pentru a obtine probabilitatile de iesire
        z2 = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        return z1, z2

    def ce_loss(self, output_true, output_computed):
        n_examples = output_true.shape[0] # numarul de exemple din setul de date
        log_probs = -np.log(output_computed[range(n_examples), output_true]) # calculam cat de sigure sunt predictiile modelului
        loss = np.sum(log_probs) / n_examples # calculam pierderea medie
        return loss

    def backward_propagation(self, features, output_true, z1, output_computed):
        n_examples = output_true.shape[0] # numarul de exemple din setul de date
        eroare2 = output_computed.copy() # copiem matricea de iesire pentru a calcula eroarea
        eroare2[range(n_examples), output_true] -= 1 # calculam eroarea pentru fiecare exemplu
        eroare2 /= n_examples # normalizam eroarea, adica facem media erorii pe toate exemplele

        gradient_w2 = np.dot(z1.T, eroare2) # calculam gradientul pentru matricea de greutati a stratului de iesire
        bias2 = np.sum(eroare2, axis=0, keepdims=True) # calculam gradientul pentru matricea de bias a stratului de iesire

        eroare1 = np.dot(eroare2, self.w2_matrix.T) # calculam eroarea pentru stratul ascuns
        eroare1[z1 <= 0] = 0 # aplicam derivata functiei de activare ReLU pentru a calcula eroarea pentru fiecare neuron din stratul ascuns

        gradient_w1 = np.dot(features.T, eroare1) # calculam gradientul pentru matricea de greutati a stratului de intrare
        bias1 = np.sum(eroare1, axis=0, keepdims=True) # calculam gradientul pentru matricea de bias a stratului de intrare

        return gradient_w1, bias1, gradient_w2, bias2

    def fit(self, features, output_corect, reg=1e-3, max_iters=1000, learning_rate=0.1):
        for i in range(max_iters):
            z1, z2 = self.forward_propagation(features)
            if i % 100 == 0:
                loss = self.ce_loss(output_corect, z2)
                reg_loss = 0.5 * reg * (np.sum(self.w1_matrix ** 2) + np.sum(self.w2_matrix ** 2)) # adaugam pierderea de regularizare pentru a preveni overfitting-ul
                print(f"Iteratia {i}, pierdere: {loss + reg_loss}")

            gradient_w1, bias1, gradient_w2, bias2 = self.backward_propagation(features, output_corect, z1, z2)
            gradient_w2 += reg * self.w2_matrix # adaugam gradientul de regularizare pentru matricea de greutati a stratului de iesire
            gradient_w1 += reg * self.w1_matrix # adaugam gradientul de regularizare pentru matricea de greutati a stratului de intrare

            self.w1_matrix -= learning_rate * gradient_w1 # actualizam matricea de greutati a stratului de intrare
            self.b1_matrix -= learning_rate * bias1 # actualizam matricea de bias
            self.w2_matrix -= learning_rate * gradient_w2 # actualizam matricea de greutati a stratului de iesire
            self.b2_matrix -= learning_rate * bias2 # actualizam matricea de bias

    def predict(self, features):
        _, output_computed = self.forward_propagation(features) # calculam iesirea modelului pentru datele de intrare
        return np.argmax(output_computed, axis=1) # returnam clasa prezisa de model pentru fiecare exemplu

In [9]:
model = ANN(features=trainFeatures.shape[1], labels=len(labelNames), hidden=64)
model.fit(trainFeatures.toarray(), np.array([labelNames.index(label) for label in trainOutputs]), reg=1e-3, max_iters=3000, learning_rate=0.01)
predictions = model.predict(testFeatures.toarray())
accuracy = 0
for i in range(len(testOutputs)):
    if labelNames[predictions[i]] == testOutputs[i]:
        accuracy += 1

print("Accuracy:", accuracy / len(testOutputs))

Accuracy: 0.65
